# Decision Tree Analysis
**Taha Furkan Torun**

# Part 1 — Spam Classification

## Data Preparation

The dataset (4,601 emails, 57 features) is split into an 80% training set and 20% test set. Spam prevalence is checked across all three subsets to confirm the split is representative.

In [16]:
install.packages("kernlab")

library(kernlab)

data("spam", package = "kernlab")

set.seed(425)

n <- nrow(spam)
train_index <- sample(1:n, size = 0.8 * n)

train_data <- spam[train_index, ]
test_data  <- spam[-train_index, ]


cat("Overall spam percentage:", mean(spam$type == "spam") * 100, "\n")
cat("Train spam percentage:", mean(train_data$type == "spam") * 100, "\n")
cat("Test spam percentage:", mean(test_data$type == "spam") * 100)





The downloaded binary packages are in
	/var/folders/b5/zg0ny5lj36n2kvl4x5b72p0m0000gn/T//RtmpJj05St/downloaded_packages
Overall spam percentage: 39.40448 
Train spam percentage: 38.42391 
Test spam percentage: 43.32248

## Building the Full Tree

A maximally grown classification tree is built with no pruning constraints. This establishes a performance baseline and shows how complex the unconstrained model becomes.

In [17]:
library(rpart)

largest_tree <- rpart(
  type ~ .,
  data = train_data,
  method = "class",
  control = rpart.control(cp = 0, minsplit = 2, minbucket = 1, xval = 10)
)

leaf_count <- sum(largest_tree$frame$var == "<leaf>")

cat("Number of leaf nodes:", leaf_count, "\n")

Number of leaf nodes: 252 


## Evaluating the Full Tree

Predictions are made on the held-out test set. Error rate, false positive rate (legitimate email flagged as spam), and false negative rate (spam that slips through) are reported.

In [18]:
pred <- predict(largest_tree, test_data, type = "class")

conf_mat <- table(Predicted = pred, Actual = test_data$type)

conf_mat

test_error <- mean(pred != test_data$type)

TP <- conf_mat["spam", "spam"]
TN <- conf_mat["nonspam", "nonspam"]
FP <- conf_mat["spam", "nonspam"]
FN <- conf_mat["nonspam", "spam"]

false_positive_rate <- FP / (FP + TN)
false_negative_rate <- FN / (FN + TP)

cat("Test error:", round(test_error, 4), "\n")
cat("False positive rate:", round(false_positive_rate, 4), "\n")
cat("False negative rate:", round(false_negative_rate, 4), "\n")

         Actual
Predicted nonspam spam
  nonspam     486   42
  spam         36  357

Test error: 0.0847 
False positive rate: 0.069 
False negative rate: 0.1053 


## Pruning — Finding the Right Tree Size

Two pruning targets are identified from the cross-validation output:

- **Minimum CV error tree:** the depth that minimizes cross-validation error
- **`opttree` (1-SE rule):** the simplest tree whose CV error stays within one standard deviation of the minimum — a more conservative choice that favors generalization

In [19]:
cp_table <- largest_tree$cptable

min_idx <- which.min(cp_table[, "xerror"])
min_cp <- cp_table[min_idx, "CP"]
min_splits <- cp_table[min_idx, "nsplit"]
min_leaves <- min_splits + 1

threshold <- cp_table[min_idx, "xerror"] + cp_table[min_idx, "xstd"]

opt_idx <- which(cp_table[, "xerror"] <= threshold)[1]
opt_cp <- cp_table[opt_idx, "CP"]
opt_splits <- cp_table[opt_idx, "nsplit"]
opt_leaves <- opt_splits + 1

opttree <- prune(largest_tree, cp = opt_cp)

cat("Tree size with minimum CV error (leaf nodes):", min_leaves, "\n")
cat("Tree size with minimum CV error (splits):", min_splits, "\n")
cat("CP for minimum CV error tree:", min_cp, "\n")

cat("Tree size of opttree (leaf nodes):", opt_leaves, "\n")
cat("Tree size of opttree (splits):", opt_splits, "\n")
cat("CP for opttree:", opt_cp, "\n")

Tree size with minimum CV error (leaf nodes): 44 
Tree size with minimum CV error (splits): 43 
CP for minimum CV error tree: 0.001414427 
Tree size of opttree (leaf nodes): 37 
Tree size of opttree (splits): 36 
CP for opttree: 0.001591231 


## Evaluating the Pruned Tree

The same metrics are computed for `opttree` and compared against the full tree.

In [20]:
pred_opt <- predict(opttree, test_data, type = "class")

conf_mat_opt <- table(Predicted = pred_opt, Actual = test_data$type)

conf_mat_opt

test_error_opt <- mean(pred_opt != test_data$type)

TP_opt <- conf_mat_opt["spam", "spam"]
TN_opt <- conf_mat_opt["nonspam", "nonspam"]
FP_opt <- conf_mat_opt["spam", "nonspam"]
FN_opt <- conf_mat_opt["nonspam", "spam"]

false_positive_rate_opt <- FP_opt / (FP_opt + TN_opt)
false_negative_rate_opt <- FN_opt / (FN_opt + TP_opt)

cat("Test error:", round(test_error_opt, 4), "\n")
cat("False positive rate:", round(false_positive_rate_opt, 4), "\n")
cat("False negative rate:", round(false_negative_rate_opt, 4), "\n")

         Actual
Predicted nonspam spam
  nonspam     491   46
  spam         31  353

Test error: 0.0836 
False positive rate: 0.0594 
False negative rate: 0.1153 


`opttree` achieves a marginally lower overall test error (8.36% vs. 8.47%) while meaningfully reducing the false positive rate — from 6.9% down to 5.94%. The trade-off is a slightly higher false negative rate (11.53% vs. 10.53%). In a real email filter, this is the preferable direction: false alarms erode user trust more than occasional missed spam.

# Part 2 — Suicide Rate Prediction

In [21]:
list.files()
suicide_data <- read.csv("suicide-rate.csv")
head(suicide_data)
str(suicide_data)

[1] "hw1.ipynb"        "suicide-rate.csv"

,country,year,sex,age,suicides_no,population,suicides.100k.pop,country.year,gdp_for_year....,gdp_per_capita....,generation
,<chr>,<int>,<chr>,<chr>,<int>,<int>,<dbl>,<chr>,<chr>,<int>,<chr>
1,Albania,1987,male,15-24 years,21,312900,6.71,Albania1987,"2,156,624,900",796,Generation X
2,Albania,1987,male,35-54 years,16,308000,5.19,Albania1987,"2,156,624,900",796,Silent
3,Albania,1987,female,15-24 years,14,289700,4.83,Albania1987,"2,156,624,900",796,Generation X
4,Albania,1987,male,75+ years,1,21800,4.59,Albania1987,"2,156,624,900",796,G.I. Generation
5,Albania,1987,male,25-34 years,9,274300,3.28,Albania1987,"2,156,624,900",796,Boomers
6,Albania,1987,female,75+ years,1,35600,2.81,Albania1987,"2,156,624,900",796,G.I. Generation


'data.frame':	27820 obs. of  11 variables:
 $ country           : chr  "Albania" "Albania" "Albania" "Albania" ...
 $ year              : int  1987 1987 1987 1987 1987 1987 1987 1987 1987 1987 ...
 $ sex               : chr  "male" "male" "female" "male" ...
 $ age               : chr  "15-24 years" "35-54 years" "15-24 years" "75+ years" ...
 $ suicides_no       : int  21 16 14 1 9 1 6 4 1 0 ...
 $ population        : int  312900 308000 289700 21800 274300 35600 278800 257200 137500 311000 ...
 $ suicides.100k.pop : num  6.71 5.19 4.83 4.59 3.28 2.81 2.15 1.56 0.73 0 ...
 $ country.year      : chr  "Albania1987" "Albania1987" "Albania1987" "Albania1987" ...
 $ gdp_for_year....  : chr  "2,156,624,900" "2,156,624,900" "2,156,624,900" "2,156,624,900" ...
 $ gdp_per_capita....: int  796 796 796 796 796 796 796 796 796 796 ...
 $ generation        : chr  "Generation X" "Silent" "Generation X" "G.I. Generation" ...


In [22]:
suicide_data$gdp_for_year <- as.numeric(gsub(",", "", suicide_data$gdp_for_year))
suicide_data2 <- suicide_data[, !(names(suicide_data) %in% c("country.year"))]

## Data Preparation

The dataset covers global suicide records from 1985 to 2016 across 101 countries (27,820 observations). The `country.year` column is dropped as it duplicates information already captured by `country` and `year` separately. The dataset is split 80/20 into training and test sets.

In [23]:
set.seed(425)

n <- nrow(suicide_data2)
train_idx <- sample(1:n, size = 0.8 * n)

train_data2 <- suicide_data2[train_idx, ]
test_data2  <- suicide_data2[-train_idx, ]
cat("Training set size:", nrow(train_data2), "\n")
cat("Test set size:", nrow(test_data2), "\n")

Training set size: 22256 
Test set size: 5564 


## Building and Pruning the Regression Tree

A regression tree is fit to predict suicides per 100k population. The tree is grown without constraints, then the optimal size is identified via cross-validation error minimization.

In [24]:
library(rpart)

tree2 <- rpart(
  suicides.100k.pop ~ .,
  data = train_data2,
  method = "anova",
  control = rpart.control(cp = 0)
)

cp_table2 <- tree2$cptable

min_idx2 <- which.min(cp_table2[, "xerror"])

min_splits2 <- cp_table2[min_idx2, "nsplit"]
min_leaves2 <- min_splits2 + 1

cat("Number of splits:", min_splits2, "\n")
cat("Number of leaf nodes:", min_leaves2, "\n")

Number of splits: 1466 
Number of leaf nodes: 1467 


## Variable Importance

The model's variable importance scores show which features drive the predictions most.

In [25]:
tree2$variable.importance

gdp_for_year....                age            country                sex 
        3886717.30         2472647.21         2169463.07         1623498.00 
        generation        suicides_no         population       gdp_for_year 
        1299269.17         1227445.43          380744.09          238810.72 
gdp_per_capita....               year 
         231830.98           46002.67

GDP for year and age group are by far the strongest predictors. Country remains significant even after accounting for GDP — pointing to structural, cultural, or policy-level differences that economic output alone does not explain. Year carries almost no signal, suggesting that the variation in this dataset is better explained by who and where than by when.

## Model Validation — CV Error vs. Test Error

In [26]:
pred2 <- predict(tree2, test_data2)

test_mse <- mean((pred2 - test_data2$suicides.100k.pop)^2)

min_xerror2 <- cp_table2[min_idx2, "xerror"]

cat("Test MSE:", test_mse, "\n")
cat("Minimum CV error:", min_xerror2, "\n")

Test MSE: 68.2222 
Minimum CV error: 0.2084494 


The CV error (0.208, relative scale) and test MSE (68.2) are on different scales and cannot be compared directly. What matters is whether the model selected by CV performs well on unseen data — and the test MSE confirms it does, with no sign of the severe overfitting seen in the unpruned tree (1,467 leaf nodes).